# レッスン09 - メタ認知デザインパターン


## セットアップ

このノートブックは、Microsoft Agent Framework を使用したメタ認知デザインパターンを示します。

**前提条件:**
- 環境変数を通じて設定された Azure OpenAI のデプロイメント
- Azure CLI で認証済み（`az login`）


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## メタ認知とは何ですか？

メタ認知とは<strong>思考について考えること</strong>です。AIエージェントの文脈では、次のことができるエージェントを構築することを意味します：

- 自分自身の出力や推論プロセスを<strong>自己反省</strong>する
- <strong>エラーを検出</strong>し、静かに失敗するのではなく優雅に回復する
- 自分の応答が完全で役に立つかを<strong>評価</strong>する
- 最初のアプローチがうまくいかない場合に戦略を<strong>適応</strong>させる（例：バックアップシステムに切り替える）

メタ認知エージェントは単に質問に答えるだけでなく、自身のパフォーマンスを監視し、その場で調整します。


## プライマリおよびバックアップツール

一般的なメタ認知パターンとして、<strong>フォールバック戦略</strong>があります。エージェントはまずプライマリーツールを試み、失敗した場合（例：404エラー）、失敗を認識してバックアップツールに透明に切り替えます。

これは、プライマリーサービスが利用できない場合に、エージェントが自己診断し代替経路を選択する現実世界のシステムを反映しています。

下記に2つのフライト検索ツールを定義します：
- <strong>プライマリー</strong> — パリ、東京、バルセロナをカバー
- <strong>バックアップ</strong> — ベルリン、シドニー、ニューヨーク市をカバー


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## エラー回復機能付き自己反省エージェント

以下のエージェントは、まず主要な飛行システムを試し、障害を認識し、透明性をもってバックアップシステムに切り替えるように指示されています。各応答の後に、その回答がユーザーの質問に完全に応えたかどうかを簡単に自己評価します。


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 自己評価パターン

メタ認知のもう一つの側面は<strong>自己評価</strong>です。これは別のエージェント（または同じエージェントが二回目に）応答の完全性、正確性、有用性を評価します。

以下では、旅行代理店の応答を3つの軸で採点する `ResponseEvaluator` エージェントを作成します。


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## サマリー

このレッスンでは、Microsoft Agent Framework を使用して<strong>メタ認知エージェント</strong>を構築する方法を学びました:

- <strong>自己反省</strong>: 自身の推論を監視し、何が起きたかを透明に伝えるエージェント。
- <strong>フォールバックによるエラー回復</strong>: 主要＋バックアップツールパターンで、エージェントが失敗（例えば404エラー）を検出し、自動的に別のソースを試す。
- <strong>自己評価</strong>: 回答の完全性、正確性、有用性を評価する別の評価者エージェント。

これらのパターンにより、エージェントはより堅牢で透明性が高く、信頼できるものとなり、実際の運用で重要な特性となります。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
